In [8]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, ModelSettings, function_tool, trace

dotenv.load_dotenv()

True

In [9]:
# Connect to existing collection
chroma_client = chromadb.PersistentClient(path="./chroma")
emission_db = chroma_client.get_collection(name="bitcoin_mining_emissions")

In [11]:
@function_tool
def emission_lookup_tool(query: str, max_results: int = 5) -> str:
    """
    Tool function for a RAG database to look up emission data collection activities 
    for Bitcoin mining operations (Scope 1 and Scope 2).

    Args:
        query: The emission activity or category to look up (e.g., "electricity", "ASIC miners", "cooling systems").
        max_results: The maximum number of results to return.

    Returns:
        A string containing the emission tracking information.
    """

    results = emission_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No emission tracking information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope = metadata["emission_scope"].upper()
        category = metadata["activity_category"].replace("_", " ").title()
        data_to_collect = metadata["data_to_collect"]
        unit = metadata["unit_of_measure"]
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()

        formatted_results.append(
            f"""
{i+1}. {emission_source}
   - Scope: {scope}
   - Category: {category}
   - Data to collect: {data_to_collect}
   - Unit: {unit}
   - Typical % of emissions: {percentage}%
   - Priority: {priority}
            """.strip()
        )

    return "Emission Tracking Activities:\n\n" + "\n\n".join(formatted_results)

In [12]:
@function_tool
def emission_filter_tool(
    scope: str = None, 
    priority_level: str = None, 
    min_percentage: float = None,
    max_results: int = 10
) -> str:
    """
    Tool function to filter emission activities by specific criteria.

    Args:
        scope: Filter by emission scope (e.g., "scope1", "scope2").
        priority_level: Filter by priority (e.g., "critical", "high", "medium", "low").
        min_percentage: Filter by minimum percentage impact (e.g., 1.0 for activities >1%).
        max_results: The maximum number of results to return.

    Returns:
        A string containing filtered emission tracking information.
    """
    
    # Build where clause
    where_clause = {}
    
    if scope:
        where_clause["emission_scope"] = scope.lower()
    
    if priority_level:
        where_clause["priority_level"] = priority_level.lower()
    
    if min_percentage is not None:
        where_clause["percentage_min"] = {"$gte": min_percentage}
    
    # Build query text based on filters
    query_parts = []
    if scope:
        query_parts.append(f"{scope} emissions")
    if priority_level:
        query_parts.append(f"{priority_level} priority")
    if min_percentage:
        query_parts.append(f"high impact activities")
    
    query_text = " ".join(query_parts) if query_parts else "emission activities"
    
    results = emission_db.query(
        query_texts=[query_text], 
        n_results=max_results,
        where=where_clause if where_clause else None
    )

    if not results["documents"][0]:
        return f"No emission activities found matching the criteria."

    # Format results
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope_label = metadata["emission_scope"].upper()
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()
        unit = metadata["unit_of_measure"]

        formatted_results.append(
            f"{i+1}. {emission_source} ({scope_label}) - {percentage}% impact - Priority: {priority} - Unit: {unit}"
        )

    return "Filtered Emission Activities:\n\n" + "\n".join(formatted_results)

In [13]:
emission_agent = Agent(
    name="Bitcoin Mining Emissions Assistant",
    instructions="""
    You are a specialized assistant helping with carbon accounting and emissions tracking 
    for Bitcoin mining operations. You provide guidance on Scope 1 and Scope 2 emission 
    data collection activities according to the GHG Protocol.

    You give clear, actionable answers about:
    - What emission data to collect
    - How to measure different emission sources
    - Priority levels for different activities
    - The typical impact (%) of different emission sources
    
    Key facts to remember:
    - Scope 1 = Direct emissions (combustion, fugitive emissions)
    - Scope 2 = Indirect emissions from purchased electricity
    - ASIC miner electricity typically represents 95-98% of total emissions (CRITICAL priority)
    - Cooling systems are also significant (1-3% of emissions)
    
    When users ask about emission tracking:
    1. Use emission_lookup_tool for general queries or specific emission sources
    2. Use emission_filter_tool when users want activities filtered by scope, priority, or impact level
    
    Always mention the priority level and typical percentage impact for activities you discuss.
    Be concise but include all essential technical details.
    """,
    tools=[emission_lookup_tool, emission_filter_tool],
    model_settings=ModelSettings(tool_choice="emission_lookup_tool")
)

In [ ]:
# Example 2: Scope-specific query
with trace("RAG - Scope 2 Query"):
    result = await Runner.run(
        emission_agent,
        "What data do I need to collect for Scope 2 emissions? List all activities with their units."
    )
    print(result.final_output)

Below are the Scope 2 data collection activities (purchased electricity and related energy services), with the data to collect and the units you should use. I’ve included typical priority and emission-impact context for each.

1) Administrative Office Electricity
- Data to collect: electricity usage for admin spaces
- Unit: kWh
- Scope: SCOPE2
- Priority: MEDIUM
- Typical emissions impact: 0.05–0.2%

2) Facility Operations Electricity
- Data to collect: total electricity usage for facility operations
- Unit: kWh
- Scope: SCOPE2
- Priority: HIGH
- Typical emissions impact: 0.1–0.5%

3) Cooling System Electricity
- Data to collect: electricity used by cooling systems (CRACs, chillers, cooling towers)
- Unit: kWh
- Scope: SCOPE2
- Priority: CRITICAL
- Typical emissions impact: 1–3%

4) ASIC Miner Electricity Consumption
- Data to collect: electricity used by all ASIC mining units
- Unit: kWh
- Scope: SCOPE2
- Priority: CRITICAL
- Typical emissions impact: 95–98%

5) Power Purchase Agreeme

In [ ]:
# Example 3: Electricity-specific query
with trace("RAG - Electricity Tracking"):
    result = await Runner.run(
        emission_agent,
        "How should I track electricity consumption for ASIC miners and cooling systems?"
    )
    print(result.final_output)

Here’s a clear, actionable way to track electricity consumption for ASIC miners and cooling systems, aligned with Scope 2 (purchased electricity) and the priority guidance you provided.

What to track (data to collect)
- ASIC miners
  - Subsystem: per-rack or per-group of miners (ideally one submeter per circuit or per 20–50 miners)
  - Data: real-time or interval power draw (kW), total energy (kWh) over the period, device/rack IDs, uptime/downtime
  - Source: sub-meters, smart PDUs, miner firmware power readouts (cross-check with meter readings)
  - Emissions: kWh × grid emission factor (kg CO2e/kWh)
  - Priority: CRITICAL
  - Typical impact: 95–98% of total mining emissions
- Cooling systems
  - Subsystem: per cooling unit (CRAC units, chillers, pumps, fans) or per zone
  - Data: power draw (kW), energy (kWh), unit IDs, cooling supply temperature if available
  - Source: cooling plant meters, HVAC submeters
  - Emissions: kWh × grid emission factor
  - Priority: CRITICAL
  - Typical 

In [16]:
# Example 4: High-impact activities
with trace("High Impact Activities"):
    result = await Runner.run(
        emission_agent,
        "Show me all emission sources that represent more than 1% of total emissions"
    )
    print(result.final_output)

Here are the emission sources that (in the data you provided) represent more than 1% of total emissions:

- ASIC Miner Electricity Consumption
  - Scope: Scope 2
  - Category: Purchased electricity
  - Typical % of emissions: 95-98%
  - Priority: CRITICAL
  - Data to collect: electricity usage (kWh)

- Cooling System Electricity
  - Scope: Scope 2
  - Category: Purchased electricity
  - Typical % of emissions: 1-3%
  - Priority: CRITICAL
  - Data to collect: electricity usage (kWh)

- Natural Gas For On-Site Power Generation
  - Scope: Scope 1
  - Category: Stationary combustion
  - Typical % of emissions: 0.1-2%
  - Priority: LOW-MEDIUM
  - Data to collect: gas consumption (cubic meters)

Notes:
- The ASIC miner electricity share is by far the dominant source (CRITICAL priority).
- Cooling systems are also significant (1-3%), deserving close monitoring (CRITICAL).
- On-site natural gas generation can exceed 1% in some setups, but is generally smaller than electricity for ASICs and coo

In [18]:
# Example 5: Fugitive emissions
with trace("Fugitive Emissions"):
    result = await Runner.run(
        emission_agent,
        "What refrigerant and fugitive emissions do I need to track for my cooling systems?"
    )
    print(result.final_output)

Short answer:
- Track refrigerant quantities by type and all fugitive emissions from cooling systems (these are Scope 1). This includes leaks, service venting, and end-of-life releases. Cooling-energy (Scope 2) is separate but contributes to total emissions.

What to track exactly
- Refrigerant inventory
  - Refrigerant type and chemical identity (e.g., R-134a, R-410A, R-32, R-404A, NH3, etc.)
  - System charge by type (kg)
  - Changes to charge (recharges, replacements, additions) with dates and amounts

- Fugitive emissions from cooling equipment
  - Annual leaked quantity by refrigerant type (kg)
  - Leak rate by system (percentage of charge leaked per year, if available)
  - Detection method and frequency (LDAR program, handheld detectors, infrared cameras, etc.)
  - Locations and equipment involved (e.g., chillers, condensers, immersion cooling loops, cooling towers)

- Service and end-of-life events
  - Maintenance events that involve opening/venting, purge, or recharging
  - Amo

In [ ]:
# Example 6: Priority-based query
with trace("Critical Activities Only"):
    result = await Runner.run(
        emission_agent,
        "What are the critical priority activities I must track? Include percentage impact."
    )
    print(result.final_output)

In [19]:
# Example 7: Complete Scope 1 overview
with trace("Complete Scope 1"):
    result = await Runner.run(
        emission_agent,
        "Give me a complete overview of all Scope 1 emission activities for a Bitcoin mining facility, organized by category"
    )
    print(result.final_output)

Here’s a complete Scope 1 overview for a Bitcoin mining facility, organized by category. The activities come from our Scope 1 data lookup. I’ve included the data to collect, units, typical impact, and priority for each item.

Category: Stationary combustion
- Natural Gas For On-Site Power Generation
  - Scope: Scope 1
  - Data to collect: natural gas consumption
  - Unit: cubic meters (or kWh)
  - Typical % of emissions: 0.1–2%
  - Priority: LOW–MEDIUM
  - Measurement approach: Record monthly natural gas use; apply region-specific emission factors for natural gas to convert to CO2e.

- Natural Gas For Facility Heating
  - Scope: Scope 1
  - Data to collect: gas consumption
  - Unit: cubic meters (or kWh)
  - Typical % of emissions: 0.05–0.3%
  - Priority: LOW
  - Measurement approach: Track gas for heating season; convert to CO2e with appropriate factors.

- Diesel Fuel For Backup Generators
  - Scope: Scope 1
  - Data to collect: diesel consumption
  - Unit: liters
  - Typical % of em

In [17]:
# Example 8: Metrics and KPIs
with trace("Performance Metrics"):
    result = await Runner.run(
        emission_agent,
        "What critical metrics should I track for operational efficiency? Include PUE and renewable energy percentage."
    )
    print(result.final_output)

Here are the critical metrics to track for operational efficiency in Bitcoin mining, with PUE and renewable energy percentage highlighted. Each item includes how to measure it, its priority, and its typical emission impact.

1) ASIC electricity efficiency (W per TH/s or J/TH)
- What to track: Power consumption per unit of mining capacity (e.g., watts per TH/s).
- How to measure: Collect per-miner or per-rack energy data and normalize by live hash rate; update daily/weekly.
- Priority: CRITICAL
- Why it matters: ASIC electricity is the dominant emission source (roughly 95–98% of total emissions). Small gains here yield large emission reductions.
- Typical impact: Directly drives Scope 2 emissions and overall CO2e; improvements scale with the energy share of total emissions.

2) Total electricity consumption (kWh) per period
- What to track: Total energy consumed by all mining hardware and supporting electrical loads.
- How to measure: Aggregate meter data from sub-mmeters and main feede